In [ ]:
import pandas as pd

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


analytics_mongo = client_db.get_database('analytics_db')

db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')

wave=app_mongo.get_database('app_db')



In [ ]:
pd.DataFrame(list(analytics_mongo.collection_names()))

# DATA AQUISATION

times=pd.read_sql("select max(epoch),min(epoch) from vehicles_metadata",con=engine)
mintime=int(times['min'].iloc[0])
maxtime=int(times['max'].iloc[0])
data=pd.DataFrame()
for i in range (mintime,maxtime,10000):
    data_reuse=pd.read_sql("select * from vehicles_metadata where epoch>{} and epoch<={}".format(i,i+10000),con=engine)
    data=pd.concat([data,data_reuse])
vehicles_done=data['Reg_No'].tolist()



# vahan data to metadata

#a=pd.read_sql("select * from vahan_data limit 10000",engine)
a=pd.DataFrame()
for i in range (0,200000,1000):
    #if 1>0:
    try:
        vehicles=pd.read_sql("select vehicle_number from vehicle_details where vehicle_id>={} and vehicle_id<{}".format(i,(i+1000)),con=engine)['vehicle_number'].tolist()      
        vehicles=[x for x in vehicles if type(x)==str]
        vehicles=[x for x in vehicles if type(x)==str]

        data_reuse=pd.read_sql("select * from vahan_data where vehicle_number in {}".format(tuple(vehicles)),con=engine)
        print(len(data_reuse))
        a=pd.concat([a,data_reuse])
    #else:
    except Exception as e:
        print("some issue: "+ str(e))
        

a.final_selected=a.final_selected.apply(literal_eval)
a.final_values=a.final_values.apply(literal_eval)
a.final_keys=a.final_keys.apply(literal_eval)
print(len(a)-len(a.vehicle_number.unique()))

a=a.drop_duplicates('vehicle_number')

columns=['Registration No:',
 'Registration Date:',
 'Chassis No:',
 'Engine No:',
 'Owner Name:',
 'Vehicle Class:',
 'Fuel:',
 'Maker / Model:',
 'Fitness/REGN Upto:',
 'MV Tax upto:',
 'Insurance Details',
 'Emission norms:',
 'RC Status:',
 "Financier's Name:",
 'Permit:',
 ' National Permit Authorization: ']
select_data=pd.DataFrame(a['final_selected'].tolist(),columns=columns)


from ast import literal_eval

a.final_selected=a.final_selected.apply(literal_eval)
a.final_values=a.final_values.apply(literal_eval)
a.final_keys=a.final_keys.apply(literal_eval)
print(len(a)-len(a.vehicle_number.unique()))

a=a.drop_duplicates('vehicle_number')

columns=['Registration No:',
 'Registration Date:',
 'Chassis No:',
 'Engine No:',
 'Owner Name:',
 'Vehicle Class:',
 'Fuel:',
 'Maker / Model:',
 'Fitness/REGN Upto:',
 'MV Tax upto:',
 'Insurance Details',
 'Emission norms:',
 'RC Status:',
 "Financier's Name:",
 'Permit:',
 ' National Permit Authorization: ']
select_data=pd.DataFrame(a['final_selected'].tolist(),columns=columns)


# DATA SANITIZATION 

In [ ]:
vahan_api_scrape=pd.read_csv("vahaan_scrape_Api.csv")
gps_vehicles=pd.read_sql("select vehicle_number from vehicles where id in (select vehicle_id from operator_vehicles where deboarding_time is null and is_active=True)",mysql_vds)      
api_scrape=vahan_api_scrape[vahan_api_scrape['Registration No'].isin(gps_vehicles['vehicle_number'])]  



In [ ]:
my_data=pd.read_csv("mydatavahan.csv")
my_data=my_data[~my_data['Registration No:'].isin(api_scrape['Registration No'])]
len(my_data)

In [ ]:
sheet_data=pd.read_excel("veh_data.xlsx")
sheet_data=sheet_data[~sheet_data['Number'].isin(api_scrape['Registration No'])]
len(sheet_data)

In [ ]:
mongo_data=pd.read_csv("mongo_data_vahan.csv")
mongo_data=mongo_data[~mongo_data['registration_no'].isin(api_scrape['Registration No'])]
len(mongo_data)

In [ ]:
my_data_list=my_data[['Registration No:','Maker / Model:']].values.tolist()

In [ ]:
sheet_list=sheet_data[['Number','Model']].values.tolist()

In [ ]:
mongo_list=mongo_data[['registration_no','maker_model']].values.tolist()

In [ ]:
vahan_api_list=api_scrape[['Registration No','Maker/Model']].values.tolist()

In [ ]:
total_list=my_data_list+sheet_list+mongo_list+vahan_api_list

In [ ]:
make_model=pd.DataFrame(total_list,columns=['vehicle_number','make/model'])

In [ ]:
make_model.vehicle_number.nunique()

In [ ]:
make_model.to_csv("make_model_176k.csv")

In [ ]:
make_model=make_model[make_model['make/model'].notnull()]

In [ ]:
make_model.drop_duplicates('vehicle_number',inplace=True)

In [ ]:
make_model['type']=make_model['make/model'].apply(lambda x: str(type(x)))

In [ ]:
make_model['type'].value_counts()

In [ ]:
make_model=make_model[make_model['type']!="<class 'datetime.datetime'>"]

In [ ]:
del make_model['type']

In [ ]:
make_model['contains_upto']=make_model['make/model'].str.findall('upto').apply(len)

In [ ]:
make_model['contains_upto'].value_counts()

In [ ]:
make_model=make_model[make_model['contains_upto']==0]

In [ ]:
del make_model['contains_upto']

In [ ]:
make_model['make']=make_model['make/model'].apply(lambda x : x.split('/')[0])


In [ ]:
make_model['contains_numbers']=make_model['make'].str.findall(r'\d\d').apply(len)

In [ ]:
make_model['contains_numbers'].value_counts()

In [ ]:
make_model=make_model[make_model['contains_numbers']==0]

In [ ]:
del make_model['contains_numbers']

In [ ]:
make_model.head(2)

In [ ]:
def split_data(x):
    try:
        a=x.split('/')[1]
        return a
    except:
        a=None
        return a

In [ ]:
make_model['model']=make_model['make/model'].apply(split_data)


In [ ]:
make_model['model'].isna().value_counts()

In [ ]:
make_model[make_model['model'].isna()]['make/model'].value_counts()

In [ ]:
make_model=make_model[make_model['model'].notnull()]

In [ ]:
make_model.head(2)

In [ ]:
make_model['model_regex']=make_model['model'].str.findall(r'(\d\d\d\d|407|ACE|\d\d.\d\d|DOST|BOLERO|BLAZO\d\d|BLAZO \d\d|709|909|912)')

  

In [ ]:
make_model['model_regex'].isna().value_counts()

In [ ]:
make_model['model_regex']=make_model['model_regex'].apply(lambda x: x if len(x)>0 else None)

In [ ]:
make_model[make_model['model_regex'].isna()].model.value_counts().keys().tolist()[0:20]

In [ ]:
make_model

In [ ]:
make_model.head()

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [ ]:
make_model.make.value_counts()[0:10]

In [ ]:
def OEM_categorisation(x):
    name_list=[ 'ASHOK LEYLAND',
        'TATA',
 'VE COMMERCIAL',
 'MAHINDRA',
 'EICHER',
 'DAIMLER',
 'SML ISUZU',
 'VECTRA',
 'Baldev',
 'FORCE',
 'PIAGGIO',
 'MAN TRUCKS',
 'AMW',
 'GENERAL']
    for i in name_list:
        if fuzz.partial_ratio(x,i)>70:
            #print('compared ' + i + ' with '+ x +' and SELECTED')
            return i 
        else:
            None
            #print('compared ' + i + ' with '+ x +' and rejected')
    return x
                
    
    

In [ ]:
make_model=make_model[~make_model['make'].isin([' ','  ','   ','    ','     '])]

In [ ]:
make_model['sanitized_make']=make_model.make.apply(OEM_categorisation)

In [ ]:
make_model

#make_numbers=pd.DataFrame()
#make_numbers['sanitized_make']=make_model['sanitized_make'].value_counts().keys().tolist()
#make_numbers['values']=make_model['sanitized_make'].value_counts().values.tolist()
#make_numbers.to_csv("OEMs_count.csv")

make_model['model_regex']=make_model['model_regex'].apply(lambda x: x if len(x)>0 else None)

In [ ]:
select_makes=make_model[make_model['sanitized_make'].isin(['MAHINDRA','TATA','ASHOK LEYLAND','VE COMMERCIAL','EICHER'])][make_model['model_regex'].notnull()]       



In [ ]:
len(select_makes)

In [ ]:
def list_string_merge(x):
    y=''
    for i in x:
        
        y+=" " + i
    return y

In [ ]:
select_makes['len_of_regex']=select_makes['model_regex'].apply(len)

In [ ]:
select_makes['model_list_merged']=select_makes['model_regex'].apply(list_string_merge)

In [ ]:
import re
select_makes['removing_dot']=select_makes.model_list_merged.apply(lambda x: re.sub('\.', '', x) )
select_makes['removing_dot']=select_makes.removing_dot.apply(lambda x: re.sub('-', '', x) )

In [ ]:
select_makes.head(2)

In [ ]:
complete_list_of_models=select_makes['removing_dot'].value_counts().keys().tolist()
list_of_models_spare=complete_list_of_models
len(list_of_models_spare)

In [ ]:
fuzzy_replacement={}
for i in list_of_models_spare:  
    for j in list_of_models_spare:
        if i!=j:
            if fuzz.partial_ratio(j,i)>90:
                #print('compared :' + j +' with :'+ i+ ' and took ****' + i +' **** as the superset' )
                #final_list_models.append(i)
                #list_of_models_spare.remove(i)
                #list_of_models_spare.remove(j)
                #print(len(list_of_models_spare))
                if j not in fuzzy_replacement.keys():
                    fuzzy_replacement.update({j:i})
        else:
            None
            #print('same')
    list_of_models_spare.remove(i)
    



In [ ]:
fuzzy_replacement

def final_models_categorisation(x):
    global final_list_models
    for i in final_list_models:
        if fuzz.partial_ratio(x,i)>90:
            return i
        else:
            None
        return x
        
    

In [ ]:
#select_makes['final_models']=select_makes['removing_dot'].apply(final_models_categorisation)
select_makes['final_models']=select_makes['removing_dot'].replace(fuzzy_replacement)


In [ ]:
select_makes[select_makes['final_models']==' 3718 6600']

In [ ]:
select_makes['final_models']=select_makes.final_models.apply(lambda x: re.sub('\.', '', x) )
select_makes['final_models']=select_makes.final_models.apply(lambda x: re.sub('-', '', x) )
select_makes['final_models']=select_makes.final_models.apply(lambda x: x[1:] )

In [ ]:
select_makes['state_of_reg']=select_makes['vehicle_number'].apply(lambda x: x[:2])

In [ ]:
correction=pd.read_csv("model_correction.csv")

In [ ]:
correction_values=dict(correction[['final_models','Correction']].values.tolist())

In [ ]:
correction_values.update({'ACE ACE':'ACE'})

In [ ]:
select_makes['corrected_model']=select_makes['final_models'].replace(correction_values)

In [ ]:
len(select_makes.corrected_model.value_counts())

In [ ]:
select_makes['complete_make_model']=select_makes.apply(lambda row: str(row['sanitized_make'])+' '+ str(row['corrected_model']),axis=1)

In [ ]:
select_makes.vehicle_number.nunique()

In [ ]:
select_makes

In [ ]:
select_makes['make_model_final']=select_makes['sanitized_make']+select_makes['corrected_model']

In [ ]:
select_makes.to_csv("data/make_model.csv")    